# Modelo predictivo de voto en la Asamblea General de la ONU
Este notebook construye un primer modelo para predecir el voto de un país en una resolución nueva usando:

- Historial de votación del país
- Coherencia histórica con países referentes
- Metadatos de la resolución (tema, agenda, sesión, año)

Nota: el dataset no incluye explícitamente patrocinadores de resolución, por lo que se usa la información disponible en `subjects`, `agenda_title` y `resolution` como metadatos del texto.

In [2]:
# Cargar librerías y datos
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

csv_path = '2026_02_06_ga_voting.csv'
df = pd.read_csv(csv_path, parse_dates=['date'], low_memory=False)
print('Filas totales:', len(df))
print('Columnas:', df.columns.tolist())

Filas totales: 947434
Columnas: ['undl_id', 'ms_code', 'ms_name', 'ms_vote', 'date', 'session', 'resolution', 'draft', 'committee_report', 'meeting', 'title', 'agenda_title', 'subjects', 'vote_note', 'total_yes', 'total_no', 'total_abstentions', 'total_non_voting', 'total_ms', 'undl_link']


In [3]:
# Preparar variables objetivo y metadatos
def map_vote(vote):
    if vote == 'Y':
        return 'Yes'
    if vote == 'N':
        return 'No'
    if vote == 'A':
        return 'Abstain'
    return 'Other'

df['vote_label'] = df['ms_vote'].apply(map_vote)

# Categorizar temas según subjects y agenda_title

def categorize_topic(row):
    text = str(row['subjects']).upper() + ' ' + str(row['agenda_title']).upper()
    if 'HUMAN RIGHTS' in text or 'RIGHTS' in text:
        return 'Human Rights'
    if 'NUCLEAR' in text or 'DISARMAMENT' in text:
        return 'Nuclear'
    if 'PEACE' in text or 'SECURITY' in text or 'WAR' in text:
        return 'Peace and Security'
    if 'DEVELOPMENT' in text or 'ECONOMIC' in text or 'SOCIAL' in text:
        return 'Development'
    if 'PALESTINE' in text or 'MIDDLE EAST' in text or 'ISRAEL' in text:
        return 'Middle East'
    return 'Other'

df['topic'] = df.apply(categorize_topic, axis=1)

df['year'] = df['date'].dt.year
# Label encode session because puede no ser numérico
session_encoder = LabelEncoder()
df['session_code'] = session_encoder.fit_transform(df['session'].astype(str))

# Países referentes para coherencia histórica
reference_countries = ['UNITED STATES', 'RUSSIAN FEDERATION', 'CHINA', 'FRANCE', 'UNITED KINGDOM']

# Calcular proporciones históricas de voto por país
country_stats = df.groupby('ms_name')['vote_label'].value_counts(normalize=True).unstack(fill_value=0).reset_index()
country_stats.columns.name = None
country_stats = country_stats.rename(columns={'Yes': 'country_yes_pct', 'No': 'country_no_pct', 'Abstain': 'country_abstain_pct'})

# Calcular la coherencia histórica con países referentes
agreements = []
for ref in reference_countries:
    if ref in df['ms_name'].unique():
        merged = df.merge(df[df['ms_name'] == ref][['undl_id', 'vote_label']], on='undl_id', suffixes=('', '_ref'))
        merged = merged[merged['ms_name'] != ref]
        merged['agree'] = (merged['vote_label'] == merged['vote_label_ref']).astype(int)
        agreement = merged.groupby('ms_name')['agree'].mean().reset_index().rename(columns={'agree': f'agree_with_{ref.replace(" ", "_")}'})
        agreements.append(agreement)

if agreements:
    agreement_features = agreements[0]
    for extra in agreements[1:]:
        agreement_features = agreement_features.merge(extra, on='ms_name', how='outer')
    agreement_features = agreement_features.fillna(0)
else:
    agreement_features = pd.DataFrame({'ms_name': df['ms_name'].unique()})

# Unir estadísticas de país y coherencia referencial
country_features = country_stats.merge(agreement_features, on='ms_name', how='left').fillna(0)

# Features de metadatos de resolución
resolution_features = df[['undl_id', 'topic', 'year', 'session_code', 'resolution']].drop_duplicates().set_index('undl_id')

# Unir todo en un dataset de entrenamiento
model_df = df[['undl_id', 'ms_name', 'vote_label']].merge(country_features, on='ms_name', how='left')
model_df = model_df.merge(resolution_features, left_on='undl_id', right_index=True, how='left')

# Codificar tópicos y país
topic_encoder = LabelEncoder()
model_df['topic_code'] = topic_encoder.fit_transform(model_df['topic'])

country_encoder = LabelEncoder()
model_df['country_code'] = country_encoder.fit_transform(model_df['ms_name'])

# Definir variables de entrenamiento
features = [
    'country_code',
    'country_yes_pct',
    'country_no_pct',
    'country_abstain_pct',
    'agree_with_UNITED_STATES',
    'agree_with_RUSSIAN_FEDERATION',
    'agree_with_CHINA',
    'agree_with_FRANCE',
    'agree_with_UNITED_KINGDOM',
    'topic_code',
    'year',
    'session_code'
]

# Filtrar datos completos
model_df = model_df.dropna(subset=features + ['vote_label'])
X = model_df[features]
y = model_df['vote_label']

test_size = 0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print('Reporte de clasificación:')
print(classification_report(y_test, y_pred, zero_division=0))
print('Matriz de confusión:')
print(confusion_matrix(y_test, y_pred))

Reporte de clasificación:
              precision    recall  f1-score   support

     Abstain       0.35      0.12      0.18     20217
          No       0.45      0.22      0.29      9397
       Other       0.73      0.51      0.60     17615
         Yes       0.81      0.95      0.87    142258

    accuracy                           0.78    189487
   macro avg       0.59      0.45      0.49    189487
weighted avg       0.74      0.78      0.75    189487

Matriz de confusión:
[[  2390    905    439  16483]
 [   781   2037    153   6426]
 [   308    117   9013   8177]
 [  3260   1511   2779 134708]]


## Conclusiones del modelo predictivo

- El modelo logra una precisión global razonable (`~78%`), pero la capacidad de predecir votos minoritarios como `Abstain` y `No` es más baja. Esto es esperado dada la fuerte tendencia de la ONU hacia `Yes` en muchas resoluciones.
- Las características usadas incluyen:
  - Historial del país (`country_yes_pct`, `country_no_pct`, `country_abstain_pct`)
  - Coherencia histórica con referentes (`EE.UU.`, `Rusia`, `China`, `Francia`, `Reino Unido`)
  - Metadatos de resolución (`topic`, `session`, `year`)
- El modelo muestra que la información histórica de país y la coherencia con referentes son útiles para predecir la dirección del voto, especialmente para `Yes`.
- Próximos pasos sugeridos:
  - usar técnicas de muestreo para equilibrar clases
  - incluir más metadatos de resolución si se dispone de patrocinadores o región temática
  - evaluar modelos adicionales como `XGBoost` o `LightGBM`


## Mejoras para aumentar precisión en clases minoritarias (No, Abstain)

Aplicaremos tres estrategias:
1. **Class weights**: Penalizar más los errores en clases minoritarias
2. **SMOTE**: Sobremuestreo sintético para balancear clases
3. **Ajuste de parámetros**: Profundidad y número de árboles optimizados

In [ ]:
# Estrategia 1: Usar class weights para penalizar errores en clases minoritarias
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: w for i, w in zip(np.unique(y_train), class_weights)}

print('Pesos de clases:')
for cls, weight in class_weight_dict.items():
    print(f'  {cls}: {weight:.2f}')

# Entrenar modelo con class weights
clf_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, max_depth=20, min_samples_split=5)
clf_balanced.fit(X_train, y_train)

y_pred_balanced = clf_balanced.predict(X_test)

print('\n=== Modelo con class weights ===')
print(classification_report(y_test, y_pred_balanced, zero_division=0))
print('Matriz de confusión:')
print(confusion_matrix(y_test, y_pred_balanced))